In [3]:
"""Imports, configuración, selectores y clase del scraper de exito.com."""
from __future__ import annotations

import logging
import re
import time
from dataclasses import dataclass
from typing import Optional

from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

import pandas as pd
from dataclasses import asdict
import requests

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("exito-scraper")

BASE_URL = "https://www.exito.com"
DEFAULT_TIMEOUT = 25
USER_AGENT = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/124.0.0.0 Safari/537.36"
)

# exito.com usa FastStore (Next.js). La grilla de productos es una lista <ul>
# cuyos <li> son las tarjetas. La clase lleva un hash que puede cambiar, por eso
# usamos un selector de atributo estable con la clase hasheada como respaldo.
SEL_GALERIA = (
    "ul[class*='product-grid_fs-product-grid'] li, "
    ".product-grid_fs-product-grid___qKN2 li"
)
# Selectores internos de cada tarjeta (verificados contra el DOM real).
SEL_NOMBRE = "h3.styles_name__qQJiK, [data-fs-product-card-title], h3"
SEL_PRECIO = "[data-fs-product-card-prices], [data-fs-container-price-otros]"
SEL_LINK = "a[data-testid='product-link'], a[href]"
SEL_IMG = "img"


@dataclass
class Producto:
    """Representa un producto extraído del catálogo."""

    nombre: str
    precio: Optional[str] = None          # precio final (a pagar)
    precio_lista: Optional[str] = None    # precio antes de descuento
    url: Optional[str] = None
    imagen: Optional[str] = None


class ExitoScraper:
    """Encapsula un navegador Selenium para scrapear exito.com."""

    def __init__(self, headless: bool = True, timeout: int = DEFAULT_TIMEOUT):
        self.timeout = timeout
        self.driver = self._crear_driver(headless)
        self.wait = WebDriverWait(self.driver, timeout)

    def _crear_driver(self, headless: bool) -> webdriver.Chrome:
        """Configura y devuelve una instancia de Chrome WebDriver."""
        options = Options()
        if headless:
            options.add_argument("--headless=new")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("--disable-gpu")
        options.add_argument("--window-size=1920,1080")
        options.add_argument(f"--user-agent={USER_AGENT}")
        options.add_argument("--disable-blink-features=AutomationControlled")
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option("useAutomationExtension", False)

        driver = webdriver.Chrome(options=options)
        driver.execute_cdp_cmd(
            "Page.addScriptToEvaluateOnNewDocument",
            {
                "source": (
                    "Object.defineProperty(navigator, 'webdriver', "
                    "{get: () => undefined})"
                )
            },
        )
        return driver

    def abrir(self, url: str = BASE_URL) -> None:
        """Navega a una URL y espera a que cargue el body."""
        logger.info("Abriendo %s", url)
        self.driver.get(url)
        self.wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))

    def buscar(self, termino: str) -> str:
        """Busca un término y espera a que aparezca la grilla de productos."""
        url = f"{BASE_URL}/s?q={termino.replace(' ', '%20')}"
        self.abrir(url)
        self._aceptar_cookies()
        try:
            self.wait.until(
                EC.presence_of_element_located((By.CSS_SELECTOR, SEL_GALERIA))
            )
        except TimeoutException:
            logger.warning("No apareció la grilla; puede que cambiara el selector")
        self._scroll_para_cargar()
        return url

    def _aceptar_cookies(self) -> None:
        """Cierra el banner de cookies si aparece."""
        selectores = [
            (By.ID, "onetrust-accept-btn-handler"),
            (By.CSS_SELECTOR, "button[aria-label*='aceptar' i]"),
        ]
        for by, selector in selectores:
            try:
                boton = WebDriverWait(self.driver, 5).until(
                    EC.element_to_be_clickable((by, selector))
                )
                boton.click()
                logger.info("Banner de cookies cerrado")
                return
            except TimeoutException:
                continue

    def _scroll_para_cargar(self, pasos: int = 6, pausa: float = 1.2) -> None:
        """Hace scroll progresivo para forzar la carga perezosa (lazy load)."""
        altura_previa = 0
        for _ in range(pasos):
            self.driver.execute_script(
                "window.scrollTo(0, document.body.scrollHeight);"
            )
            time.sleep(pausa)
            altura_actual = self.driver.execute_script(
                "return document.body.scrollHeight"
            )
            if altura_actual == altura_previa:
                break
            altura_previa = altura_actual

    def extraer_productos(self) -> list[Producto]:
        """Extrae los productos de la página de resultados actual."""
        productos: list[Producto] = []
        tarjetas = self.driver.find_elements(By.CSS_SELECTOR, SEL_GALERIA)
        logger.info("Se encontraron %d tarjetas", len(tarjetas))

        for tarjeta in tarjetas:
            nombre = self._texto_opcional(tarjeta, SEL_NOMBRE)
            if not nombre:
                continue
            precio, precio_lista = self._parsear_precios(
                self._texto_opcional(tarjeta, SEL_PRECIO)
            )
            productos.append(
                Producto(
                    nombre=nombre,
                    precio=precio,
                    precio_lista=precio_lista,
                    url=self._atributo_opcional(tarjeta, SEL_LINK, "href"),
                    imagen=self._atributo_opcional(tarjeta, SEL_IMG, "src"),
                )
            )

        logger.info("Se extrajeron %d productos", len(productos))
        return productos

    def diagnostico_tarjeta(self, indice: int = 0) -> str:
        """Devuelve el HTML de una tarjeta para inspeccionar sus selectores."""
        tarjetas = self.driver.find_elements(By.CSS_SELECTOR, SEL_GALERIA)
        if not tarjetas:
            return "Sin tarjetas: revisa SEL_GALERIA."
        return tarjetas[indice].get_attribute("outerHTML")

    @staticmethod
    def _parsear_precios(texto: Optional[str]) -> tuple[Optional[str], Optional[str]]:
        """Separa el texto de precios en (precio_final, precio_lista).

        El bloque puede venir como '$ 52.900' o, con descuento,
        '-15%\\n$ 21.380\\n$ 18.173' (descuento, precio lista, precio final).
        """
        if not texto:
            return None, None
        montos = re.findall(r"\$\s?[\d.,]+", texto)
        montos = [m.replace(" ", "") for m in montos]
        if not montos:
            return None, None
        if len(montos) == 1:
            return montos[0], None
        # Con descuento: el último es el precio a pagar; el primero, el de lista.
        return montos[-1], montos[0]

    @staticmethod
    def _texto_opcional(elemento, selector: str) -> Optional[str]:
        try:
            texto = elemento.find_element(By.CSS_SELECTOR, selector).text.strip()
            return texto or None
        except NoSuchElementException:
            return None

    @staticmethod
    def _atributo_opcional(elemento, selector: str, atributo: str) -> Optional[str]:
        try:
            valor = elemento.find_element(
                By.CSS_SELECTOR, selector
            ).get_attribute(atributo)
            return valor.strip() if valor else None
        except NoSuchElementException:
            return None

    def cerrar(self) -> None:
        """Cierra el navegador y libera recursos."""
        if self.driver:
            self.driver.quit()
            logger.info("Navegador cerrado")

    def __enter__(self) -> "ExitoScraper":
        return self

    def __exit__(self, *_) -> None:
        self.cerrar()

# Cierra una sesión previa del navegador si existe (evita instancias huérfanas)
_previo = globals().get("scraper")
if _previo is not None:
    try:
        _previo.cerrar()
    except Exception:
        pass

scraper = ExitoScraper(headless=True)
scraper.buscar("arroz")
productos = scraper.extraer_productos()

for producto in productos:

    requests.post(
        "http://127.0.0.1:8000/productos",
        json=asdict(producto)
    )

df = pd.DataFrame([asdict(prod) for prod in productos])
print(f"Total productos: {len(df)}")
df.head(20)


20:47:52 [INFO] Navegador cerrado
20:47:53 [INFO] Abriendo https://www.exito.com/s?q=arroz
20:48:09 [INFO] Se encontraron 16 tarjetas
20:48:09 [INFO] Se extrajeron 16 productos


Total productos: 16


,nombre,precio,precio_lista,url,imagen
0,Supremo Selección Especial Arroba SUPREMO Semi...,$52.900,NaN,https://www.exito.com/supremo-seleccion-especi...,https://exitocol.vtexassets.com/arquivos/ids/3...
1,Arroz DIANA blanco vitamor (5000 gr),$18.173,$21.380,https://www.exito.com/arroz-diana-5000-gr-4986...,https://exitocol.vtexassets.com/arquivos/ids/3...
2,Arroz ROA blanco fortiplus (3000 gr),$12.100,NaN,https://www.exito.com/arroz-blanco-fortificado...,https://exitocol.vtexassets.com/arquivos/ids/3...
3,Arroz EXITO MARCA PROPIA blanco (5000 gr),$14.960,NaN,https://www.exito.com/arroz-blanco-exito-marca...,https://d3ez54m90carx6.cloudfront.net/CUCARDA-...
4,Arroz DIANA blanco premium (2500 gr),$13.685,$17.000,https://www.exito.com/arroz-premium-2500-gr-21...,https://exitocol.vtexassets.com/arquivos/ids/3...
5,Arroz EXITO MARCA PROPIA blanco (2500 gr),$7.480,$7.490,https://www.exito.com/arroz-blanco-exito-marca...,https://d3ez54m90carx6.cloudfront.net/CUCARDA-...
6,Arroz EXITO MARCA PROPIA blanco arroba (12500 gr),$38.750,NaN,https://www.exito.com/arroz-blanco-arroba-exit...,https://d3ez54m90carx6.cloudfront.net/CUCARDA-...
7,Arroz CASTELLANO blanco premium (2500 gr),$21.080,$26.350,https://www.exito.com/arroz-blanco-natural-bol...,https://exitocol.vtexassets.com/arquivos/ids/3...
8,Arroz FLOR HUILA blanco (12500 gr),$50.200,NaN,https://www.exito.com/arroz-arroba-flor-huila-...,https://exitocol.vtexassets.com/arquivos/ids/3...
9,Arroz DIANA blanco premium (1000 gr),$5.193,$6.110,https://www.exito.com/arroz-premium-1000-gr-13...,https://exitocol.vtexassets.com/arquivos/ids/3...
